# Day 3 | Reporting and ML Handoff

This notebook supports the final day of the module.

## Focus
- visualisation outputs for a leadership pack
- transparent exception logic
- final exports
- feature engineering for downstream ML work
- written interpretation and handoff notes

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs/day3_pack")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

customers = pd.read_csv(DATA_DIR / "customers.csv", dtype={"customer_id": "string"})
accounts = pd.read_csv(
    DATA_DIR / "accounts.csv",
    dtype={"account_id": "string", "customer_id": "string"},
)
transactions = pd.read_csv(
    DATA_DIR / "transactions.csv",
    dtype={"account_id": "string", "branch_id": "string"},
)
branches = pd.read_csv(DATA_DIR / "branches.csv", dtype={"branch_id": "string"})

transactions["txn_ts"] = pd.to_datetime(transactions["txn_ts"], errors="coerce")
transactions["amount_sar"] = pd.to_numeric(transactions["amount_sar"], errors="coerce")
transactions["fee_sar"] = pd.to_numeric(transactions["fee_sar"], errors="coerce")

analysis_df = (
    transactions.merge(accounts[["account_id", "customer_id", "product_family"]], on="account_id", how="left")
    .merge(customers[["customer_id", "region", "segment"]], on="customer_id", how="left")
    .merge(branches[["branch_id", "city"]], on="branch_id", how="left")
)
analysis_df["month"] = analysis_df["txn_ts"].dt.to_period("M").astype(str)
analysis_df.head()

In [ ]:
posted = analysis_df[analysis_df["status"] == "POSTED"].copy()
monthly = (
    posted.groupby("month")
    .agg(txn_count=("txn_id", "count"), total_fee_sar=("fee_sar", "sum"))
    .reset_index()
)

branch_fees = (
    posted.groupby("branch_id")
    .agg(total_fee_sar=("fee_sar", "sum"))
    .reset_index()
    .sort_values("total_fee_sar", ascending=False)
    .head(5)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(monthly["month"], monthly["txn_count"], marker="o")
axes[0].set_title("Monthly transaction count")
axes[0].tick_params(axis="x", rotation=45)
axes[1].bar(branch_fees["branch_id"], branch_fees["total_fee_sar"])
axes[1].set_title("Top branches by fee")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "pack_charts.png", dpi=150)

branch_month = (
    posted.groupby(["branch_id", "month"])
    .agg(txn_count=("txn_id", "count"), total_fee_sar=("fee_sar", "sum"))
    .reset_index()
)
branch_month["fee_change"] = branch_month.groupby("branch_id")["total_fee_sar"].diff()
exceptions = branch_month.loc[branch_month["fee_change"].fillna(0).abs() > 2.5].copy()
exceptions["reason_code"] = "fee_change_gt_2_5"
exceptions.to_csv(OUTPUT_DIR / "exceptions.csv", index=False)

monthly

In [ ]:
cut_off_date = pd.Timestamp("2026-06-30")
window_df = posted.loc[posted["txn_ts"] <= cut_off_date].copy()
window_df["is_digital"] = window_df["channel"].fillna("").str.upper().isin(["MOB", "ONLINE"])

features = (
    window_df.groupby("customer_id")
    .agg(
        txn_count_30d=("txn_id", "count"),
        avg_ticket_30d=("amount_sar", "mean"),
        digital_share_30d=("is_digital", "mean"),
        total_fee_30d=("fee_sar", "sum"),
        product_count=("product_family", "nunique"),
        region=("region", "first"),
        segment=("segment", "first"),
    )
    .reset_index()
)
features["cut_off_date"] = cut_off_date.date().isoformat()
features.to_csv(OUTPUT_DIR / "features.csv", index=False)

assumptions = pd.DataFrame(
    [
        {"topic": "cut_off_date", "note": "Features only use data available on or before 2026-06-30."},
        {"topic": "digital_share_30d", "note": "Computed from transactions with channel values MOB or ONLINE."},
        {"topic": "exceptions", "note": "Rule based flags are illustrative and for training only."},
    ]
)
assumptions.to_csv(OUTPUT_DIR / "assumptions.csv", index=False)

features

## Day 3 Exercises

### Pack outputs
- save one chart image and one summary table
- make each chart support a specific analytical claim
- create an `exceptions.csv` file with transparent reasons and defensible thresholds

### Handoff outputs
- export a feature table
- add an assumptions file or a small data dictionary
- state one plausible modelling use for each major feature
- write two sentences for a management pack, including one confidence limit or caveat

### Final reminder
A correct answer is stronger when the caveats are visible.
An elite answer is stronger again when the scope limits, threshold choices, and alternative explanations are visible too.